# High-Throughput Batch Processing with Nemotron 3 Super

This notebook demonstrates how to run **high-throughput batch inference** with `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B` using [vLLM](https://docs.vllm.ai).

Nemotron 3 Super achieves 5× throughput over the previous Nemotron Super and 7.5× over Qwen3.5-122B on long-output workloads. This cookbook shows how to unlock that throughput for real batch workloads like document classification, summarization, and data extraction.

**What you'll learn:**
1. Configuring vLLM for maximum throughput (CUTLASS backend, EP, batch sizing)
2. Offline batch inference with vLLM's `LLM` class
3. Concurrent server requests with async Python
4. Practical use case: bulk document classification with structured JSON output
5. Measuring and comparing throughput across configurations

For model details, see the [model card](https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16). For latency-optimized single-request usage, see [vllm_cookbook.ipynb](./vllm_cookbook.ipynb).

Prerequisites:
- NVIDIA GPU with recent drivers (>= 264 GB VRAM for BF16, >= 160 GB VRAM for FP8, >= 80 GB VRAM for NVFP4) and CUDA 12.x.
- Python 3.10+

## Install dependencies

In [ ]:
#If pip not found
!python -m ensurepip --default-pip

In [ ]:
%pip install -U vllm==0.17.1 torch==2.10.0 flashinfer-python==0.6.4 flashinfer-cubin==0.6.4 'nvidia-cutlass-dsl>=4.4.0.dev1' openai httpx --extra-index-url https://download.pytorch.org/whl/cu128

## Verify GPU

Confirm CUDA is available and your GPU is visible to PyTorch.

In [ ]:
# GPU environment check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")

## Part 1: Throughput-Optimized Server Configuration

vLLM supports two MoE backend modes for Nemotron 3 Super:

| Mode | Env Var | Backend | Best For |
|------|---------|---------|----------|
| **Latency** | `VLLM_FLASHINFER_MOE_BACKEND=latency` | TRT-LLM Gen kernels | Online serving, interactive chat |
| **Throughput** | `VLLM_FLASHINFER_MOE_BACKEND=throughput` | CUTLASS | Offline batch jobs, bulk processing |

The `throughput` backend trades per-request latency for higher aggregate tokens/sec across many concurrent requests.

### Latency-optimized (default from vllm_cookbook)

```bash
VLLM_FLASHINFER_MOE_BACKEND=latency \
vllm serve nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16 \
  --tensor-parallel-size 4 \
  --trust-remote-code \
  --gpu-memory-utilization 0.9 \
  --enable-expert-parallel
```

### Throughput-optimized (for batch workloads)

```bash
VLLM_FLASHINFER_MOE_BACKEND=throughput \
VLLM_USE_FLASHINFER_MOE_FP8=1 \
vllm serve nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16 \
  --tensor-parallel-size 4 \
  --trust-remote-code \
  --gpu-memory-utilization 0.95 \
  --enable-expert-parallel \
  --max-num-seqs 256 \
  --disable-log-requests
```

Key differences for throughput:
- `VLLM_FLASHINFER_MOE_BACKEND=throughput` selects CUTLASS kernels optimized for batch processing
- `--gpu-memory-utilization 0.95` allocates more memory for KV cache (more concurrent sequences)
- `--max-num-seqs 256` allows more concurrent sequences in a batch
- `--disable-log-requests` reduces logging overhead during high-volume processing

See the [Advanced Deployment Guide](./AdvancedDeploymentGuide/README.md) for GPU-specific configurations (GB200, B200, DGX Spark).

## Part 2: Offline Batch Inference

For bulk processing without running a server, use vLLM's `LLM` class directly.
This is the simplest way to process a large dataset: load the model once, pass all prompts, collect results.

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
import time

# Choose your precision/GPU configuration
model_id = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16"
# model_id = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8"
# model_id = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-NVFP4"

llm = LLM(
    model=model_id,
    trust_remote_code=True,
    dtype="auto",
    tensor_parallel_size=4,  # TP=4 for BF16, TP=2 for FP8, TP=1 for NVFP4
    kv_cache_dtype="fp8",
    gpu_memory_utilization=0.95,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
print("Model ready for batch inference")

### Prepare a batch of prompts

We'll generate a sample dataset of 50 documents to classify. In practice, this would be your CSV, JSONL, or database rows.

In [ ]:
# Sample documents for classification
sample_documents = [
    "A new CRISPR-based gene therapy shows promise in treating sickle cell disease in clinical trials.",
    "Tesla stock surged 8% after reporting record quarterly deliveries of 500,000 vehicles.",
    "Researchers at MIT developed a neural network that can predict protein folding 100x faster.",
    "The Federal Reserve held interest rates steady, signaling a wait-and-see approach to inflation.",
    "SpaceX successfully launched its 50th Starlink mission this year, deploying 60 satellites.",
    "A longitudinal study found that Mediterranean diets reduce cardiovascular risk by 30%.",
    "NVIDIA announced its next-generation Blackwell GPU architecture at GTC 2026.",
    "The European Central Bank cut rates by 25 basis points to stimulate economic growth.",
    "Google DeepMind published a breakthrough in mathematical theorem proving using LLMs.",
    "Apple's Vision Pro headset sold 1 million units in its first quarter of availability.",
    "New evidence suggests that gut microbiome diversity is linked to cognitive performance.",
    "Amazon Web Services reported 35% year-over-year growth in cloud computing revenue.",
    "A team at Stanford created a robotic system that can perform autonomous surgery.",
    "Bitcoin reached a new all-time high above $150,000 amid institutional adoption.",
    "Moderna announced a combined flu-COVID vaccine with 90% efficacy in Phase 3 trials.",
    "Meta released its latest open-source model, achieving state-of-the-art on coding benchmarks.",
    "Climate scientists reported that 2025 was the hottest year in recorded history.",
    "Toyota unveiled a solid-state battery EV with 900-mile range for 2027 production.",
    "A new antibiotic compound was discovered that kills drug-resistant bacteria in mice.",
    "Microsoft Azure announced native support for NVIDIA NIM microservices.",
]

# Duplicate to create a larger batch (adjust multiplier for your workload)
documents = sample_documents * 5  # 100 documents
print(f"Total documents to classify: {len(documents)}")

In [ ]:
# Build classification prompts
SYSTEM_PROMPT = """You are a document classifier. For each document, output exactly one JSON object with these fields:
- "category": one of ["Science", "Technology", "Finance", "Health", "Business"]
- "confidence": a float between 0 and 1
- "summary": a one-sentence summary (max 20 words)

Output ONLY the JSON object, no other text."""

prompts = [
    tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Classify this document:\n\n{doc}"},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    for doc in documents
]

print(f"Prepared {len(prompts)} prompts")
print(f"\nExample prompt (first 200 chars):\n{prompts[0][:200]}...")

### Run batch inference and measure throughput

In [ ]:
params = SamplingParams(
    temperature=0.0,  # Deterministic for classification
    max_tokens=150,   # JSON output is compact
)

# Run the batch
start = time.perf_counter()
outputs = llm.generate(prompts, sampling_params=params)
elapsed = time.perf_counter() - start

# Calculate throughput metrics
total_output_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
total_input_tokens = sum(len(o.prompt_token_ids) for o in outputs)

print(f"Batch size:          {len(prompts)}")
print(f"Wall-clock time:     {elapsed:.2f}s")
print(f"Total input tokens:  {total_input_tokens:,}")
print(f"Total output tokens: {total_output_tokens:,}")
print(f"Output tok/s:        {total_output_tokens / elapsed:.1f}")
print(f"Docs/sec:            {len(prompts) / elapsed:.1f}")
print(f"Avg time/doc:        {elapsed / len(prompts) * 1000:.1f}ms")

In [ ]:
import json

# Parse and display results
results = []
parse_errors = 0

for i, output in enumerate(outputs):
    text = output.outputs[0].text.strip()
    try:
        parsed = json.loads(text)
        results.append(parsed)
    except json.JSONDecodeError:
        parse_errors += 1
        results.append({"raw": text, "error": "parse_failed"})

print(f"Successfully parsed: {len(results) - parse_errors}/{len(results)}")
print(f"Parse errors: {parse_errors}")
print()

# Show first 5 results
for i in range(min(5, len(results))):
    print(f"Doc {i+1}: {documents[i][:80]}...")
    print(f"  -> {json.dumps(results[i])}")
    print()

### Release GPU memory before server mode

The offline LLM object holds GPU memory. Release it before starting a vLLM server in the next section.

In [ ]:
import gc

del llm
del tokenizer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("GPU memory released. Ready for server mode.")

## Part 3: Concurrent Server Requests

For production workloads, you'll typically run vLLM as a server and send requests concurrently.
This is useful when:
- Your data arrives as a stream (files, database rows, API calls)
- You need to integrate with existing pipelines
- You want to control concurrency independently from model batching

### Start the server

Run this in a terminal before continuing:

```bash
VLLM_FLASHINFER_MOE_BACKEND=throughput \
VLLM_USE_FLASHINFER_MOE_FP8=1 \
vllm serve nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16 \
  --tensor-parallel-size 4 \
  --trust-remote-code \
  --gpu-memory-utilization 0.95 \
  --enable-expert-parallel \
  --max-num-seqs 256 \
  --disable-log-requests \
  --port 5000
```

Wait until you see `Uvicorn running on http://0.0.0.0:5000` before running the next cells.

In [ ]:
import asyncio
import httpx
import json
import time
from typing import Any

SERVER_URL = "http://127.0.0.1:5000/v1/chat/completions"
MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16"

SYSTEM_PROMPT = """You are a document classifier. For each document, output exactly one JSON object with these fields:
- "category": one of ["Science", "Technology", "Finance", "Health", "Business"]
- "confidence": a float between 0 and 1
- "summary": a one-sentence summary (max 20 words)

Output ONLY the JSON object, no other text."""


async def classify_one(
    client: httpx.AsyncClient,
    document: str,
    semaphore: asyncio.Semaphore,
) -> dict[str, Any]:
    """Send a single classification request."""
    async with semaphore:
        payload = {
            "model": MODEL_NAME,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Classify this document:\n\n{document}"},
            ],
            "temperature": 0.0,
            "max_tokens": 150,
        }
        t0 = time.perf_counter()
        resp = await client.post(SERVER_URL, json=payload, timeout=120.0)
        latency = time.perf_counter() - t0

        data = resp.json()
        text = data["choices"][0]["message"]["content"]
        usage = data.get("usage", {})

        return {
            "text": text,
            "latency": latency,
            "prompt_tokens": usage.get("prompt_tokens", 0),
            "completion_tokens": usage.get("completion_tokens", 0),
        }


async def batch_classify(
    documents: list[str],
    concurrency: int = 32,
) -> list[dict[str, Any]]:
    """Classify all documents concurrently with bounded concurrency."""
    semaphore = asyncio.Semaphore(concurrency)
    async with httpx.AsyncClient() as client:
        tasks = [
            classify_one(client, doc, semaphore)
            for doc in documents
        ]
        return await asyncio.gather(*tasks)


print("Batch classification functions defined.")

### Run concurrent classification

In [ ]:
# Reuse the same sample documents from Part 2
sample_documents = [
    "A new CRISPR-based gene therapy shows promise in treating sickle cell disease in clinical trials.",
    "Tesla stock surged 8% after reporting record quarterly deliveries of 500,000 vehicles.",
    "Researchers at MIT developed a neural network that can predict protein folding 100x faster.",
    "The Federal Reserve held interest rates steady, signaling a wait-and-see approach to inflation.",
    "SpaceX successfully launched its 50th Starlink mission this year, deploying 60 satellites.",
    "A longitudinal study found that Mediterranean diets reduce cardiovascular risk by 30%.",
    "NVIDIA announced its next-generation Blackwell GPU architecture at GTC 2026.",
    "The European Central Bank cut rates by 25 basis points to stimulate economic growth.",
    "Google DeepMind published a breakthrough in mathematical theorem proving using LLMs.",
    "Apple's Vision Pro headset sold 1 million units in its first quarter of availability.",
    "New evidence suggests that gut microbiome diversity is linked to cognitive performance.",
    "Amazon Web Services reported 35% year-over-year growth in cloud computing revenue.",
    "A team at Stanford created a robotic system that can perform autonomous surgery.",
    "Bitcoin reached a new all-time high above $150,000 amid institutional adoption.",
    "Moderna announced a combined flu-COVID vaccine with 90% efficacy in Phase 3 trials.",
    "Meta released its latest open-source model, achieving state-of-the-art on coding benchmarks.",
    "Climate scientists reported that 2025 was the hottest year in recorded history.",
    "Toyota unveiled a solid-state battery EV with 900-mile range for 2027 production.",
    "A new antibiotic compound was discovered that kills drug-resistant bacteria in mice.",
    "Microsoft Azure announced native support for NVIDIA NIM microservices.",
]

documents = sample_documents * 5  # 100 documents

# Run batch classification
start = time.perf_counter()
results = await batch_classify(documents, concurrency=32)
total_time = time.perf_counter() - start

# Aggregate metrics
total_prompt_tokens = sum(r["prompt_tokens"] for r in results)
total_completion_tokens = sum(r["completion_tokens"] for r in results)
latencies = [r["latency"] for r in results]

print(f"Documents processed: {len(results)}")
print(f"Concurrency:         32")
print(f"Wall-clock time:     {total_time:.2f}s")
print(f"Docs/sec:            {len(results) / total_time:.1f}")
print(f"Total output tokens: {total_completion_tokens:,}")
print(f"Output tok/s:        {total_completion_tokens / total_time:.1f}")
print(f"Avg latency/req:     {sum(latencies) / len(latencies) * 1000:.0f}ms")
print(f"P50 latency:         {sorted(latencies)[len(latencies)//2] * 1000:.0f}ms")
print(f"P99 latency:         {sorted(latencies)[int(len(latencies)*0.99)] * 1000:.0f}ms")

### Effect of concurrency on throughput

Higher concurrency lets vLLM batch more requests together, improving throughput up to a point.
Beyond the server's `--max-num-seqs`, additional requests queue without benefit.

In [ ]:
concurrency_levels = [1, 4, 16, 32, 64, 128]
concurrency_results = []

for c in concurrency_levels:
    start = time.perf_counter()
    res = await batch_classify(documents[:50], concurrency=c)  # Use 50 docs per test
    elapsed = time.perf_counter() - start
    tokens = sum(r["completion_tokens"] for r in res)
    concurrency_results.append({
        "concurrency": c,
        "wall_time": elapsed,
        "docs_per_sec": 50 / elapsed,
        "tok_per_sec": tokens / elapsed,
    })
    print(f"Concurrency {c:>3d}: {elapsed:6.2f}s | {50/elapsed:5.1f} docs/s | {tokens/elapsed:6.1f} tok/s")

print("\nHigher concurrency improves throughput until the server's batch capacity is saturated.")

## Part 4: Processing Files from Disk

In real workloads, your data comes from files. Here's a pattern for processing a JSONL file
line-by-line with concurrent requests and writing results back to disk.

In [ ]:
import json
from pathlib import Path

# Create a sample input file
input_path = Path("/tmp/nemotron_batch_input.jsonl")
output_path = Path("/tmp/nemotron_batch_output.jsonl")

with open(input_path, "w") as f:
    for i, doc in enumerate(documents):
        f.write(json.dumps({"id": i, "text": doc}) + "\n")

print(f"Wrote {len(documents)} documents to {input_path}")


async def process_jsonl(
    input_file: Path,
    output_file: Path,
    concurrency: int = 32,
) -> dict:
    """Process a JSONL file: classify each document and write results."""
    # Read input
    records = []
    with open(input_file) as f:
        for line in f:
            records.append(json.loads(line))

    # Classify all documents
    texts = [r["text"] for r in records]
    start = time.perf_counter()
    results = await batch_classify(texts, concurrency=concurrency)
    elapsed = time.perf_counter() - start

    # Write output
    with open(output_file, "w") as f:
        for record, result in zip(records, results):
            try:
                classification = json.loads(result["text"])
            except json.JSONDecodeError:
                classification = {"raw": result["text"], "error": "parse_failed"}
            output = {
                **record,
                "classification": classification,
                "latency_ms": round(result["latency"] * 1000),
            }
            f.write(json.dumps(output) + "\n")

    return {
        "total_docs": len(records),
        "wall_time": elapsed,
        "docs_per_sec": len(records) / elapsed,
    }


stats = await process_jsonl(input_path, output_path, concurrency=32)
print(f"\nProcessed {stats['total_docs']} docs in {stats['wall_time']:.2f}s ({stats['docs_per_sec']:.1f} docs/s)")
print(f"Results written to {output_path}")

# Show first 3 output records
with open(output_path) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        record = json.loads(line)
        print(f"\nDoc {record['id']}: {record['text'][:60]}...")
        print(f"  -> {json.dumps(record['classification'])}")

## Part 5: Throughput Benchmarking

Use `vllm bench serve` for standardized throughput measurement.
This is the recommended way to benchmark before and after configuration changes.

### Quick benchmark against your running server

```bash
# Synthetic benchmark: fixed input/output lengths
vllm bench serve \
  --model nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16 \
  --base-url http://127.0.0.1:5000 \
  --num-prompts 100 \
  --random-input-len 512 \
  --random-output-len 256 \
  --request-rate inf
```

### Compare backend modes

Run the benchmark twice - once with each backend - to see the difference:

| Configuration | Best for | Expected throughput profile |
|--------------|----------|---------------------------|
| `VLLM_FLASHINFER_MOE_BACKEND=latency` | Interactive, low-latency serving | Lower TTFT, lower aggregate tok/s |
| `VLLM_FLASHINFER_MOE_BACKEND=throughput` | Batch jobs, offline processing | Higher aggregate tok/s, higher per-request latency |

The throughput backend can deliver significantly higher aggregate tokens/sec when many requests are in flight simultaneously,
at the cost of higher per-request latency. For batch workloads where you care about total wall-clock time (not individual response speed),
the throughput backend is the right choice.

### Scaling estimates

Use your benchmark results to estimate processing time for larger workloads:

In [ ]:
# Replace with your measured docs/sec from Part 2 or Part 3
measured_docs_per_sec = stats["docs_per_sec"]

workloads = [
    ("Small dataset", 1_000),
    ("Medium dataset", 10_000),
    ("Large dataset", 100_000),
    ("Patent-scale (3.5M)", 3_500_000),
]

print(f"Based on measured throughput: {measured_docs_per_sec:.1f} docs/sec\n")
print(f"{'Workload':<25} {'Documents':>12} {'Est. Time':>15}")
print("-" * 55)
for name, count in workloads:
    seconds = count / measured_docs_per_sec
    if seconds < 60:
        time_str = f"{seconds:.0f}s"
    elif seconds < 3600:
        time_str = f"{seconds/60:.1f} min"
    else:
        time_str = f"{seconds/3600:.1f} hours"
    print(f"{name:<25} {count:>12,} {time_str:>15}")

## Cleanup and shutdown

To free resources after this notebook:

1. Release in-process model memory (run the next cell).
2. Stop the `vllm serve` process in the terminal where it was started (`Ctrl+C`).
3. If needed, restart the kernel to ensure a clean state.

In [ ]:
import gc
import torch

# Release in-process model/tokenizer objects if present
for name in ("llm", "tokenizer"):
    if name in globals():
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# Clean up temp files
from pathlib import Path
for p in [Path("/tmp/nemotron_batch_input.jsonl"), Path("/tmp/nemotron_batch_output.jsonl")]:
    p.unlink(missing_ok=True)

print("Cleanup complete.")
print("If vllm serve is running in a terminal, stop it with Ctrl+C.")